# Лабораторная работа №3

## Методы многомерного поиска

**Вариант 2:** `a = 150`, `b = 2`, `f0 = 100`, `n = 3`.

**Цель работы:**

1. Изучить алгоритмы многомерного поиска 1-го и 2-го порядка.
2. Реализовать методы поиска минимума функции многих переменных.
3. Найти экстремум тестовой функции Розенброка.
4. Сравнить методы по числу итераций, времени работы и точности результата.

**Реализуемые методы:**

1. Метод Флетчера–Ривза.
2. Метод Полака–Рибьера.
3. Метод Девидона–Флетчера–Пауэлла.
4. Метод BFGS.
5. Метод L-BFGS.


## 1. Импорт библиотек

В данном блоке подключаются библиотеки для численных вычислений, измерения времени работы алгоритмов и построения графиков сходимости.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

np.set_printoptions(precision=6, suppress=True)

## 2. Постановка задачи

Необходимо найти минимум тестовой функции Розенброка:

$$
f(x) = \sum_{i=1}^{n-1} \left( a(x_{i+1} - x_i^2)^2 + b(x_i - 1)^2 \right) + f_0
$$

Для варианта 2:

$$
a = 150,\quad b = 2,\quad f_0 = 100,\quad n = 3
$$

Глобальный минимум функции Розенброка достигается в точке:

$$
x^* = (1, 1, 1)
$$

Значение функции в этой точке:

$$
f(x^*) = f_0 = 100
$$


In [ ]:
a = 150
b = 2
f0 = 100
n = 3

x0 = np.array([-1.2, 1.0, 0.5])

eps_grad = 1e-6
eps_x = 1e-8
eps_f = 1e-10
max_iter = 10000

print("Параметры варианта:")
print(f"a = {a}")
print(f"b = {b}")
print(f"f0 = {f0}")
print(f"n = {n}")
print(f"Начальная точка x0 = {x0}")

## 3. Реализация функции Розенброка и ее градиента

Для работы методов оптимизации необходимо определить саму функцию и ее градиент.

Градиент показывает направление наибольшего роста функции. В методах минимизации используется противоположное направление — направление антиградиента.

In [ ]:
def rosenbrock(x):
    x = np.asarray(x)
    total = 0.0
    for i in range(len(x) - 1):
        total += a * (x[i + 1] - x[i] ** 2) ** 2 + b * (x[i] - 1) ** 2
    return total + f0


def rosenbrock_grad(x):
    x = np.asarray(x)
    grad = np.zeros_like(x, dtype=float)

    for i in range(len(x) - 1):
        grad[i] += -4 * a * x[i] * (x[i + 1] - x[i] ** 2) + 2 * b * (x[i] - 1)
        grad[i + 1] += 2 * a * (x[i + 1] - x[i] ** 2)

    return grad


print("Значение функции в начальной точке:")
print(rosenbrock(x0))

print("\nГрадиент в начальной точке:")
print(rosenbrock_grad(x0))

print("\nЗначение функции в теоретической точке минимума:")
print(rosenbrock(np.ones(n)))

## 4. Метод одномерного поиска

Во всех реализуемых методах требуется выбирать шаг $\alpha_k$.

Для этого используется метод золотого сечения, который минимизирует функцию одной переменной:

$$
\varphi(\alpha) = f(x_k + \alpha d_k)
$$

где $d_k$ — направление поиска.

In [ ]:
def golden_section_search(phi, left=0.0, right=1.0, tol=1e-6, max_iter=200):
    gr = (np.sqrt(5) - 1) / 2

    c = right - gr * (right - left)
    d = left + gr * (right - left)

    fc = phi(c)
    fd = phi(d)

    for _ in range(max_iter):
        if abs(right - left) < tol:
            break

        if fc < fd:
            right = d
            d = c
            fd = fc
            c = right - gr * (right - left)
            fc = phi(c)
        else:
            left = c
            c = d
            fc = fd
            d = left + gr * (right - left)
            fd = phi(d)

    return (left + right) / 2


def line_search(f, x, direction):
    phi = lambda alpha: f(x + alpha * direction)
    right = 1.0
    f0_value = phi(0.0)

    while phi(right) < f0_value and right < 100:
        right *= 2

    return golden_section_search(phi, 0.0, right)


def make_result(name, x, iterations, elapsed_time, history):
    return {
        "Метод": name,
        "Найденная точка": x,
        "Значение функции": rosenbrock(x),
        "Норма градиента": np.linalg.norm(rosenbrock_grad(x)),
        "Количество итераций": iterations,
        "Время работы, сек": elapsed_time,
        "История": history
    }

## 5. Метод Флетчера–Ривза

Метод Флетчера–Ривза относится к методам сопряженных градиентов.

На первой итерации направление совпадает с направлением антиградиента:

$$
d_0 = - \nabla f(x_0)
$$

На следующих итерациях направление строится с учетом предыдущего направления:

$$
d_k = -\nabla f(x_k) + \beta_k d_{k-1}
$$

где

$$
\beta_k = \frac{\|\nabla f(x_k)\|^2}{\|\nabla f(x_{k-1})\|^2}
$$


In [ ]:
def fletcher_reeves(x_start):
    start_time = time.time()

    x = x_start.astype(float).copy()
    g = rosenbrock_grad(x)
    d = -g
    history = [rosenbrock(x)]

    for k in range(max_iter):
        if np.linalg.norm(g) < eps_grad:
            break

        alpha = line_search(rosenbrock, x, d)
        x_new = x + alpha * d
        g_new = rosenbrock_grad(x_new)

        beta = np.dot(g_new, g_new) / max(np.dot(g, g), 1e-20)
        d = -g_new + beta * d

        history.append(rosenbrock(x_new))

        if np.linalg.norm(x_new - x) < eps_x and abs(rosenbrock(x_new) - rosenbrock(x)) < eps_f:
            x = x_new
            g = g_new
            break

        x = x_new
        g = g_new

    elapsed_time = time.time() - start_time
    return make_result("Флетчер–Ривз", x, k + 1, elapsed_time, history)


result_fr = fletcher_reeves(x0)

print("Результат метода Флетчера–Ривза:")
print("x =", result_fr["Найденная точка"])
print("f(x) =", result_fr["Значение функции"])
print("Количество итераций =", result_fr["Количество итераций"])
print("Время =", result_fr["Время работы, сек"])

## 6. Метод Полака–Рибьера

Метод Полака–Рибьера также относится к методам сопряженных градиентов.

Отличие от метода Флетчера–Ривза состоит в формуле для коэффициента $\beta_k$:

$$
\beta_k = \frac{\nabla f(x_k)^T(\nabla f(x_k) - \nabla f(x_{k-1}))}{\|\nabla f(x_{k-1})\|^2}
$$

На практике используется модификация:

$$
\beta_k = \max(0, \beta_k)
$$


In [ ]:
def polak_ribiere(x_start):
    start_time = time.time()

    x = x_start.astype(float).copy()
    g = rosenbrock_grad(x)
    d = -g
    history = [rosenbrock(x)]

    for k in range(max_iter):
        if np.linalg.norm(g) < eps_grad:
            break

        alpha = line_search(rosenbrock, x, d)
        x_new = x + alpha * d
        g_new = rosenbrock_grad(x_new)

        beta = np.dot(g_new, g_new - g) / max(np.dot(g, g), 1e-20)
        beta = max(0, beta)

        d = -g_new + beta * d

        history.append(rosenbrock(x_new))

        if np.linalg.norm(x_new - x) < eps_x and abs(rosenbrock(x_new) - rosenbrock(x)) < eps_f:
            x = x_new
            g = g_new
            break

        x = x_new
        g = g_new

    elapsed_time = time.time() - start_time
    return make_result("Полак–Рибьер", x, k + 1, elapsed_time, history)


result_pr = polak_ribiere(x0)

print("Результат метода Полака–Рибьера:")
print("x =", result_pr["Найденная точка"])
print("f(x) =", result_pr["Значение функции"])
print("Количество итераций =", result_pr["Количество итераций"])
print("Время =", result_pr["Время работы, сек"])

## 7. Метод Девидона–Флетчера–Пауэлла

Метод DFP относится к квазиньютоновским методам.

В квазиньютоновских методах явно не вычисляется матрица Гессе. Вместо этого строится приближение обратной матрицы Гессе.

Направление поиска вычисляется по формуле:

$$
d_k = -H_k \nabla f(x_k)
$$


In [ ]:
def dfp_method(x_start):
    start_time = time.time()

    x = x_start.astype(float).copy()
    H = np.eye(len(x))
    g = rosenbrock_grad(x)
    history = [rosenbrock(x)]

    for k in range(max_iter):
        if np.linalg.norm(g) < eps_grad:
            break

        d = -H @ g

        if np.dot(d, g) >= 0:
            d = -g
            H = np.eye(len(x))

        alpha = line_search(rosenbrock, x, d)
        x_new = x + alpha * d
        g_new = rosenbrock_grad(x_new)

        s = x_new - x
        y = g_new - g

        sy = np.dot(s, y)
        Hy = H @ y
        yHy = np.dot(y, Hy)

        if abs(sy) > 1e-12 and abs(yHy) > 1e-12:
            H = H + np.outer(s, s) / sy - np.outer(Hy, Hy) / yHy

        history.append(rosenbrock(x_new))

        if np.linalg.norm(x_new - x) < eps_x and abs(rosenbrock(x_new) - rosenbrock(x)) < eps_f:
            x = x_new
            g = g_new
            break

        x = x_new
        g = g_new

    elapsed_time = time.time() - start_time
    return make_result("DFP", x, k + 1, elapsed_time, history)


result_dfp = dfp_method(x0)

print("Результат метода DFP:")
print("x =", result_dfp["Найденная точка"])
print("f(x) =", result_dfp["Значение функции"])
print("Количество итераций =", result_dfp["Количество итераций"])
print("Время =", result_dfp["Время работы, сек"])

## 8. Метод BFGS

Метод BFGS является одним из наиболее популярных квазиньютоновских методов.

Он строит приближение обратной матрицы Гессе более устойчивым способом, чем DFP.

Обновление выполняется по формуле:

$$
H_{k+1} = (I - \rho s y^T)H_k(I - \rho y s^T) + \rho ss^T
$$

где:

$$
\rho = \frac{1}{y^Ts}
$$


In [ ]:
def bfgs_method(x_start):
    start_time = time.time()

    x = x_start.astype(float).copy()
    H = np.eye(len(x))
    g = rosenbrock_grad(x)
    history = [rosenbrock(x)]

    for k in range(max_iter):
        if np.linalg.norm(g) < eps_grad:
            break

        d = -H @ g

        if np.dot(d, g) >= 0:
            d = -g
            H = np.eye(len(x))

        alpha = line_search(rosenbrock, x, d)
        x_new = x + alpha * d
        g_new = rosenbrock_grad(x_new)

        s = x_new - x
        y = g_new - g
        ys = np.dot(y, s)

        if abs(ys) > 1e-12:
            rho = 1.0 / ys
            I = np.eye(len(x))
            H = (I - rho * np.outer(s, y)) @ H @ (I - rho * np.outer(y, s)) + rho * np.outer(s, s)

        history.append(rosenbrock(x_new))

        if np.linalg.norm(x_new - x) < eps_x and abs(rosenbrock(x_new) - rosenbrock(x)) < eps_f:
            x = x_new
            g = g_new
            break

        x = x_new
        g = g_new

    elapsed_time = time.time() - start_time
    return make_result("BFGS", x, k + 1, elapsed_time, history)


result_bfgs = bfgs_method(x0)

print("Результат метода BFGS:")
print("x =", result_bfgs["Найденная точка"])
print("f(x) =", result_bfgs["Значение функции"])
print("Количество итераций =", result_bfgs["Количество итераций"])
print("Время =", result_bfgs["Время работы, сек"])

## 9. Метод L-BFGS

Метод L-BFGS является модификацией BFGS с ограниченной памятью.

В отличие от BFGS, он не хранит полную матрицу $H_k$. Вместо этого используются несколько последних пар векторов:

$$
s_k = x_{k+1} - x_k
$$

$$
y_k = \nabla f(x_{k+1}) - \nabla f(x_k)
$$

Это делает метод удобным для задач большой размерности.

In [ ]:
def lbfgs_direction(g, s_list, y_list):
    q = g.copy()
    alpha_list = []
    rho_list = []

    for s, y in reversed(list(zip(s_list, y_list))):
        rho = 1.0 / max(np.dot(y, s), 1e-20)
        alpha = rho * np.dot(s, q)
        q = q - alpha * y
        alpha_list.append(alpha)
        rho_list.append(rho)

    if len(s_list) > 0:
        s_last = s_list[-1]
        y_last = y_list[-1]
        gamma = np.dot(s_last, y_last) / max(np.dot(y_last, y_last), 1e-20)
    else:
        gamma = 1.0

    r = gamma * q

    for s, y, alpha, rho in zip(s_list, y_list, reversed(alpha_list), reversed(rho_list)):
        beta = rho * np.dot(y, r)
        r = r + s * (alpha - beta)

    return -r


def lbfgs_method(x_start, memory=5):
    start_time = time.time()

    x = x_start.astype(float).copy()
    g = rosenbrock_grad(x)
    s_list = []
    y_list = []
    history = [rosenbrock(x)]

    for k in range(max_iter):
        if np.linalg.norm(g) < eps_grad:
            break

        d = lbfgs_direction(g, s_list, y_list)

        if np.dot(d, g) >= 0:
            d = -g
            s_list = []
            y_list = []

        alpha = line_search(rosenbrock, x, d)
        x_new = x + alpha * d
        g_new = rosenbrock_grad(x_new)

        s = x_new - x
        y = g_new - g

        if np.dot(s, y) > 1e-12:
            s_list.append(s)
            y_list.append(y)

            if len(s_list) > memory:
                s_list.pop(0)
                y_list.pop(0)

        history.append(rosenbrock(x_new))

        if np.linalg.norm(x_new - x) < eps_x and abs(rosenbrock(x_new) - rosenbrock(x)) < eps_f:
            x = x_new
            g = g_new
            break

        x = x_new
        g = g_new

    elapsed_time = time.time() - start_time
    return make_result("L-BFGS", x, k + 1, elapsed_time, history)


result_lbfgs = lbfgs_method(x0)

print("Результат метода L-BFGS:")
print("x =", result_lbfgs["Найденная точка"])
print("f(x) =", result_lbfgs["Значение функции"])
print("Количество итераций =", result_lbfgs["Количество итераций"])
print("Время =", result_lbfgs["Время работы, сек"])

## 10. Сравнение результатов

В данном блоке формируется итоговая таблица, в которой сравниваются все реализованные методы.

Сравнение проводится по следующим критериям:

- найденная точка;
- значение функции;
- норма градиента;
- число итераций;
- время работы.

In [ ]:
results = [result_fr, result_pr, result_dfp, result_bfgs, result_lbfgs]

summary = pd.DataFrame([
    {
        "Метод": r["Метод"],
        "x1": r["Найденная точка"][0],
        "x2": r["Найденная точка"][1],
        "x3": r["Найденная точка"][2],
        "f(x)": r["Значение функции"],
        "Норма градиента": r["Норма градиента"],
        "Количество итераций": r["Количество итераций"],
        "Время работы, сек": r["Время работы, сек"]
    }
    for r in results
])

display(summary)

## 11. Графики сходимости методов

Для оценки скорости сходимости строятся графики зависимости значения функции от номера итерации.

Чем быстрее график приближается к минимальному значению $f_0 = 100$, тем быстрее сходится метод.

In [ ]:
plt.figure(figsize=(10, 6))

for r in results:
    values = np.array(r["История"])
    plt.plot(values - f0 + 1e-12, label=r["Метод"])

plt.yscale("log")
plt.xlabel("Номер итерации")
plt.ylabel("f(x) - f0")
plt.title("Сравнение скорости сходимости методов")
plt.legend()
plt.grid(True)
plt.show()

## 12. График сравнения количества итераций

На данном графике сравнивается количество итераций, которое потребовалось каждому алгоритму для достижения критерия остановки.

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(summary["Метод"], summary["Количество итераций"])
plt.xticks(rotation=30)
plt.ylabel("Количество итераций")
plt.title("Сравнение методов по количеству итераций")
plt.grid(axis="y")
plt.show()

## 13. График сравнения времени работы

На данном графике сравнивается время работы каждого алгоритма.

Время работы зависит не только от количества итераций, но и от сложности вычислений внутри одной итерации.

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(summary["Метод"], summary["Время работы, сек"])
plt.xticks(rotation=30)
plt.ylabel("Время, сек")
plt.title("Сравнение методов по времени работы")
plt.grid(axis="y")
plt.show()

## 14. Проверка решения на допустимость

Теоретически минимум функции Розенброка для данного варианта находится в точке:

$$
x^* = (1, 1, 1)
$$

Значение функции в этой точке:

$$
f(x^*) = 100
$$

Проверим, насколько близко найденные решения находятся к теоретическому минимуму.

In [ ]:
x_star = np.ones(n)
f_star = rosenbrock(x_star)

check = pd.DataFrame([
    {
        "Метод": r["Метод"],
        "Расстояние до x*": np.linalg.norm(r["Найденная точка"] - x_star),
        "Отклонение f(x) от f*": abs(r["Значение функции"] - f_star),
        "Норма градиента": r["Норма градиента"]
    }
    for r in results
])

display(check)

print("Теоретическая точка минимума:", x_star)
print("Теоретическое значение функции:", f_star)

## 15. Стационарная точка

Стационарная точка функции определяется условием:

$$
\nabla f(x) = 0
$$

Для функции Розенброка глобальная стационарная точка минимума находится в точке:

$$
x = (1, 1, 1)
$$

В этой точке значение функции равно:

$$
f(x) = f_0 = 100
$$


In [ ]:
stationary_point = np.ones(n)

print("Стационарная точка:")
print(stationary_point)

print("\nЗначение функции в стационарной точке:")
print(rosenbrock(stationary_point))

print("\nГрадиент в стационарной точке:")
print(rosenbrock_grad(stationary_point))

print("\nНорма градиента:")
print(np.linalg.norm(rosenbrock_grad(stationary_point)))

## 16. Итоговый вывод

В ходе лабораторной работы были реализованы и исследованы методы многомерного поиска:

1. Метод Флетчера–Ривза.
2. Метод Полака–Рибьера.
3. Метод DFP.
4. Метод BFGS.
5. Метод L-BFGS.

Для варианта 2 была рассмотрена функция Розенброка с параметрами:

$$
a = 150,\quad b = 2,\quad f_0 = 100,\quad n = 3
$$

По результатам работы все методы приблизились к теоретической точке минимума:

$$
x^* = (1,1,1)
$$

и значению функции:

$$
f(x^*) = 100
$$

На основе таблиц и графиков можно сравнить методы по числу итераций, времени работы и точности.

In [ ]:
print("Лабораторная работа №3 выполнена.")
print("Реализованы методы Флетчера–Ривза, Полака–Рибьера, DFP, BFGS и L-BFGS.")
print("Найден минимум функции Розенброка для варианта 2.")